In [1]:
WORK_DIR_PATH = ".."

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.chdir(WORK_DIR_PATH)
print(f"DIRECTORY: {os.getcwd()}")

DIRECTORY: c:\Users\jayar\Desktop\바탕 화면\REPO\PROJECT\M3-PJT_RS


In [4]:
import sys
sys.path.append("src")

# Config

In [5]:
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from pointwise import utils

# Ratings

In [29]:
# Upload Data
PATH = "./data/origin/ratings.csv"
ratings = pd.read_csv(PATH)

In [30]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [31]:
# Positive feedback is limited to ratings of 3 or higher
pos_feedback = ratings[ratings["rating"]>=3]

In [32]:
# Columns
kwargs = dict(
    df=pos_feedback,
    user_col='userId', 
    item_col='movieId', 
)

data = utils.standardizer.main(**kwargs)

In [33]:
data.head()

,userId,itemId
0,2,2
1,2,4
2,2,7
3,2,45
4,2,48


In [34]:
# data Description
kwargs = dict(
    df=data,
    percentile=0.9,
)

utils.desc.main(**kwargs)

number of user: 609
number of item: 8452
total interaction: 81763
interaction density: 1.5885 %
max interaction of user: 2117
max interaction of item: 315
top 10.0 % interaction of user: 344.4
top 10.0 % interaction of item: 25.0
mean interaction of user: 134
mean interaction of item: 9


In [13]:
PATH = "./data/ratings.csv"

kwargs = dict(
    path_or_buf=PATH,
    index=False, 
    header=True,
)

data.to_csv(**kwargs)

# Genres

In [14]:
PATH = f"./data/origin/movies.csv"
genres = pd.read_csv(PATH)

In [15]:
genres = genres["genres"].str.get_dummies(sep="|")

In [16]:
genres.head()

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [21]:
genres["PAD"] = 0
genres["OOV"] = 0

In [22]:
genres.head()

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,...,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,PAD,OOV
0,0,0,1,1,1,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
cols = ["PAD", "OOV"] + list(genres.columns[1:-2])
genres = genres[cols]

In [25]:
genres.head()

,PAD,OOV,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [26]:
kwargs = dict(
    data=genres.values, 
    dtype=torch.float32,
)

genres = torch.tensor(**kwargs)

In [27]:
genres

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [28]:
PATH = "./data/genres.pt"

kwargs = dict(
    obj=genres, 
    f=PATH,
)

torch.save(**kwargs)